In [6]:
# Step 1: Install dependencies
!pip install -q gensim numpy wordfreq


In [7]:
import random
import numpy as np
import gensim.downloader as api
from wordfreq import top_n_list


In [9]:
# Step 2: Load pre-trained GloVe embeddings (100-dimensional)
glove = api.load("glove-wiki-gigaword-100")
print(f"Loaded {len(glove):,} words with vector dimension {glove.vector_size}")


Loaded 400,000 words with vector dimension 100


In [10]:
# Step 3: Curate vocabulary & pre-normalize embedding matrix
VOCAB_SIZE = 30000

# Filter top English words present in GloVe
raw_words = top_n_list("en", VOCAB_SIZE * 2)
vocab = [w for w in raw_words if w in glove and w.isalpha() and len(w) >= 2][:VOCAB_SIZE]

# Build pre-normalized matrix for vectorized cosine similarity
vocab_matrix = np.array([glove[w] for w in vocab], dtype=np.float32)
norms = np.linalg.norm(vocab_matrix, axis=1, keepdims=True)
vocab_matrix_norm = vocab_matrix / np.maximum(norms, 1e-9)

print(f"Vocabulary ready: {len(vocab):,} words")
print(f"Matrix shape: {vocab_matrix_norm.shape}")


Vocabulary ready: 30,000 words
Matrix shape: (30000, 100)


In [11]:
class ContextoGame:
    def __init__(self, vocab, vocab_matrix_norm, model):
        self.vocab = vocab
        self.vocab_set = set(vocab)
        self.matrix_norm = vocab_matrix_norm
        self.model = model
        self.target_word = None
        self.rank_lookup = {}
        self.rank_to_word = {}
        self.guess_history = []

    def start_new_game(self, target_word=None):
        if target_word is None:
            candidates = [w for w in self.vocab[:4000] if len(w) >= 4]
            self.target_word = random.choice(candidates)
        else:
            self.target_word = target_word.lower().strip()
            if self.target_word not in self.vocab_set:
                raise ValueError(f"'{self.target_word}' is not in vocabulary.")

        # Vectorized cosine similarity: single dot product against all vocab vectors
        t_vec = self.model[self.target_word]
        t_norm = t_vec / np.linalg.norm(t_vec)
        similarities = self.matrix_norm @ t_norm
        sorted_indices = np.argsort(-similarities)

        self.rank_lookup = {}
        self.rank_to_word = {}
        for rank, idx in enumerate(sorted_indices, start=1):
            w = self.vocab[idx]
            self.rank_lookup[w] = (rank, float(similarities[idx]))
            self.rank_to_word[rank] = w

        self.guess_history = []
        return self.target_word

    def guess(self, word):
        word = word.lower().strip()
        if word not in self.vocab_set:
            return {"success": False, "error": f"'{word}' is not in vocabulary."}

        rank, sim = self.rank_lookup[word]
        tier = "green" if rank <= 300 else ("yellow" if rank <= 3000 else "red")

        result = {
            "success": True,
            "word": word,
            "rank": rank,
            "similarity": round(sim, 4),
            "tier": tier,
            "is_winner": rank == 1
        }

        if not any(g["word"] == word for g in self.guess_history):
            self.guess_history.append(result)
            self.guess_history.sort(key=lambda x: x["rank"])

        return result

    def get_hint(self, target_rank=300):
        target_rank = min(max(target_rank, 2), len(self.vocab))
        guessed = {g["word"] for g in self.guess_history}
        for offset in range(len(self.vocab)):
            for r in [target_rank - offset, target_rank + offset]:
                if 2 <= r <= len(self.vocab):
                    candidate = self.rank_to_word[r]
                    if candidate not in guessed:
                        return self.guess(candidate)
        return None

In [2]:
!pip install -q ipywidgets

In [12]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Step 5: Inline UI for VS Code / Jupyter
game = ContextoGame(vocab, vocab_matrix_norm, glove)
game.start_new_game()

# UI Widgets (render directly inside cell output)
text_box = widgets.Text(placeholder="Type a word, 'hint', or 'giveup'...", description="Guess:")
submit_btn = widgets.Button(description="Submit", button_style="primary")
hint_btn = widgets.Button(description="💡 Hint", button_style="warning")
giveup_btn = widgets.Button(description="🏳️ Give Up", button_style="danger")
output_area = widgets.Output()

def render_state(message=""):
    with output_area:
        clear_output(wait=True)
        if message:
            print(message + "\n")
            
        print("--- Closest Guesses So Far ---")
        if not game.guess_history:
            print("  No guesses yet. Start guessing!")
        else:
            for i, g in enumerate(game.guess_history[:8], start=1):
                icon = "🟢" if g["tier"] == "green" else ("🟡" if g["tier"] == "yellow" else "🔴")
                print(f"  {i}. {icon} {g['word']:<12} Rank {g['rank']:<5} (sim: {g['similarity']})")
        print("------------------------------")

def handle_guess(_):
    word = text_box.value.strip().lower()
    text_box.value = ""  # Clear input box for next guess
    if not word:
        return

    if word == "giveup":
        render_state(f"🏳️ Secret word was: '{game.target_word}' (Rank 1)")
        return
    if word == "hint":
        best = game.guess_history[0]["rank"] if game.guess_history else 1000
        hint = game.get_hint(target_rank=max(best // 2, 2))
        render_state(f"💡 Hint: '{hint['word']}' -> Rank {hint['rank']} ({hint['tier']})")
        return

    res = game.guess(word)
    if not res["success"]:
        render_state(f"⚠️ {res['error']}")
        return

    icon = "🟢" if res["tier"] == "green" else ("🟡" if res["tier"] == "yellow" else "🔴")
    if res["is_winner"]:
        render_state(f"🎉 Solved in {len(game.guess_history)} unique guesses! '{res['word']}' was Rank 1! 🏆")
    else:
        render_state(f"{icon} '{res['word']}' -> Rank {res['rank']} (similarity: {res['similarity']})")

def handle_hint(_):
    best = game.guess_history[0]["rank"] if game.guess_history else 1000
    hint = game.get_hint(target_rank=max(best // 2, 2))
    render_state(f"💡 Hint: '{hint['word']}' -> Rank {hint['rank']} ({hint['tier']})")

def handle_giveup(_):
    render_state(f"🏳️ Secret word was: '{game.target_word}' (Rank 1)")

# Attach event handlers
text_box.on_submit(handle_guess)
submit_btn.on_click(handle_guess)
hint_btn.on_click(handle_hint)
giveup_btn.on_click(handle_giveup)

# Display widgets inline in the cell output
controls = widgets.HBox([text_box, submit_btn, hint_btn, giveup_btn])
display(controls, output_area)
render_state("🎮 New game started! Type your guess and press Enter.")

Output()